# Notebook 04: Structured Response Extraction and Validation

**Project:** `ofphil-classification-v2`  
**Phase:** 05 — Model Selection and Training (`06_llm_interview_assistant`)  
**Objective:** Implement schema-constrained JSON extraction from patient interview transcripts with strict separation between patient claims and caseworker verifications.

---

## 1. Research Objectives & Pydantic Schema Definition

The extraction engine transforms qualitative dialogues into a verifiable clinical data structure. Crucially:
1. **Preserves Original Patient Verbatim**: Every numeric or categorical claim maps back to the exact quote.
2. **Represents Missing Data as `null`**: Never replaces missing values with zero or assumptions.
3. **Separates Patient Disclosures from MSW Observations**: What the patient stated is distinct from what the social worker verified.

In [ ]:
# Pydantic Canonical Interview Dossier Schema
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
import json

class ExtractedEntity(BaseModel):
    field_name: str
    value: Optional[Any] = None
    raw_quote: str = Field(..., description="Verbatim quote from the patient utterance")
    confidence_score: Optional[float] = Field(None, ge=0.0, le=1.0)
    status: str = Field("PATIENT_REPORTED_UNVERIFIED", description="Verification status")

class SocioeconomicExtractionDossier(BaseModel):
    patient_anon_id: str
    # Core patient-disclosed fields (null if unclarified)
    monthly_family_income: Optional[float] = None
    income_modality: Optional[str] = Field(None, description="'formal_regular', 'informal_variable', 'pension', 'none'")
    household_size: Optional[int] = None
    number_of_dependents: Optional[int] = None
    employment_status: Optional[str] = None
    residence_tenure: Optional[str] = None
    monthly_rent_expense: Optional[float] = None
    monthly_utility_expense: Optional[float] = None
    has_philhealth: Optional[bool] = None
    
    # Verbatim audit trail
    evidence_quotes: List[ExtractedEntity] = []
    
    # Social Worker Review & Verification Block (Filled only by human MSW)
    msw_observations: List[str] = []
    msw_flagged_for_investigation: List[str] = []
    msw_verification_status: str = "PENDING_HUMAN_REVIEW"

print("Canonical Pydantic extraction schema defined successfully.")

## 2. Ollama JSON-Constrained Extraction Client Harness

This function leverages Ollama's `format: 'json'` enforcement to extract fields conforming to our Pydantic schema.

In [ ]:
# JSON extraction engine using local Ollama
import requests

EXTRACTION_PROMPT = """You are the OFPHIL MSW Structured Extraction Engine.
Analyze the following interview transcript snippet and extract all factual socioeconomic data into JSON.

TRANSCRIPT:
{transcript}

EXTRACTION RULES:
1. Extract ONLY facts explicitly stated by the patient.
2. If an amount or number is NOT stated, output null. DO NOT guess or assume.
3. Capture the exact verbatim quote for each extracted fact in 'evidence_quotes'.
4. Output strictly valid JSON matching this schema:
{
  "monthly_family_income": null or float,
  "income_modality": null or "formal_regular" or "informal_variable" or "none",
  "household_size": null or integer,
  "number_of_dependents": null or integer,
  "employment_status": null or string,
  "residence_tenure": null or string,
  "has_philhealth": null or boolean,
  "evidence_quotes": [
    {"field_name": "string", "value": "string/number", "raw_quote": "string"}
  ]
}
"""

def extract_structured_dossier_via_ollama(
    transcript: str,
    model_tag: str = "qwen3:8b",
    endpoint: str = "http://localhost:11434"
) -> Dict[str, Any]:
    """
    Sends transcript to local Ollama with temperature=0.0 and JSON schema formatting.
    """
    payload = {
        "model": model_tag,
        "prompt": EXTRACTION_PROMPT.format(transcript=transcript),
        "format": "json",
        "stream": False,
        "options": {
            "temperature": 0.0, # Deterministic extraction
            "max_tokens": 1024
        }
    }
    try:
        resp = requests.post(f"{endpoint}/api/generate", json=payload, timeout=60)
        if resp.status_code == 200:
            return json.loads(resp.json().get('response', '{}'))
        else:
            return {"error": f"HTTP {resp.status_code}: {resp.text}"}
    except requests.exceptions.RequestException as e:
        return {"offline_fallback": True, "message": f"Ollama offline: {e}"}

print("Extraction client configured.")

## 3. Schema Validation & Verbatim Quote Verification Demonstration

We demonstrate parsing, Pydantic type validation, and quote verification against a synthetic interview transcript.

In [ ]:
# Synthetic demonstration: Validating extracted dictionary into Pydantic Dossier
synthetic_raw_extraction = {
    "monthly_family_income": None,  # Preserved as None because daily 200-300 was irregular
    "income_modality": "informal_variable",
    "household_size": 4,
    "number_of_dependents": 2,
    "employment_status": "Self-Employed (Tricycle Driver)",
    "residence_tenure": "Shared_Living_with_Relatives",
    "has_philhealth": True,
    "evidence_quotes": [
        {
            "field_name": "employment_status",
            "value": "Self-Employed (Tricycle Driver)",
            "raw_quote": "Namasada po ako ng traysikel dati ma'am",
            "status": "PATIENT_REPORTED_UNVERIFIED"
        },
        {
            "field_name": "household_size",
            "value": 4,
            "raw_quote": "Apat po kaming mag-anak na nakatira sa bahay ng biyenan ko",
            "status": "PATIENT_REPORTED_UNVERIFIED"
        }
    ]
}

# Instantiate validated dossier
dossier = SocioeconomicExtractionDossier(
    patient_anon_id="SYN-PAT-0042",
    **synthetic_raw_extraction
)

# Simulate MSW Review and Verification
dossier.msw_observations.append("Applicant reports informal driving; earnings are irregular and below statutory minimum wage.")
dossier.msw_flagged_for_investigation.append("Require barangay certificate of indigency to corroborate informal residency with in-laws.")
dossier.msw_verification_status = "IN_PROGRESS_AWAITING_BARANGAY_CERT"

print("Validated Dossier JSON:")
print(dossier.model_dump_json(indent=2))